# Website Crawler Lab

The goal of this lab is to try different data ingestion strategies to the determine the one with great scores

In [ ]:
!uv pip install playwright beautifulsoup4
!playwright install chromium

In [ ]:
import os
import asyncio
import hashlib
import urllib.robotparser
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import nest_asyncio
import lxml

try:
    BS_PARSER = "lxml"
except ImportError:
    BS_PARSER = "html.parser"

print(f"BeautifulSoup parser: {BS_PARSER}")

In [ ]:
load_dotenv(override=True)

nest_asyncio.apply()

In [ ]:
# SITE = "https://www.veroliq.com"
SITE = "https://www.topfaith.edu.ng"

### Async Page Fetcher

In [ ]:
from playwright.async_api import async_playwright

USER_AGENT = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

async def fetch_page(context, url: str, timeout_ms: int = 30_000) -> dict:
    """Fetch a single page with Playwright; returns html + title. Skips non-HTML assets."""
    page = await context.new_page()
    try:
        response = await page.goto(url, wait_until="networkidle", timeout=timeout_ms)
        status = response.status if response else None
        content_type = response.headers.get("content-type", "") if response else ""

        # Skip binary assets (PDFs, images, etc.) — no point rendering them
        if not content_type.startswith("text/html"):
            return {"html": "", "title": "", "status": status, "skipped": True}

        html = await page.content()
        title = await page.title()
        return {"html": html, "title": title, "status": status, "skipped": False}
    finally:
        await page.close()

In [ ]:
CONCURRENCY_LIMIT = 5  # default — overridable per crawl_site call
POLITENESS_DELAY = 0.5
MAX_RETRIES = 3
RETRY_BACKOFF = 2

### Link Extraction

In [ ]:
import re
from datetime import datetime


def normalize_url(url: str) -> str:
    """Strip trailing slash (except root), drop fragment."""
    p = urlparse(url)
    path = p.path.rstrip("/") or "/"
    return p._replace(path=path, fragment="").geturl()


def extract_links(html: str, current_url: str, base_url: str) -> set[str]:
    """
    Parse all <a href> links from html, return only same-domain absolute URLs.
    Normalizes trailing slashes to prevent duplicate crawls.
    """
    base_domain = urlparse(base_url).netloc
    soup = BeautifulSoup(html, BS_PARSER)
    links = set()
    for tag in soup.find_all("a", href=True):
        href = tag["href"].strip()
        if not href or href.startswith("mailto:") or href.startswith("tel:"):
            continue
        full_url = urljoin(current_url, href)
        parsed = urlparse(full_url)
        if parsed.netloc == base_domain and parsed.scheme in ("http", "https"):
            links.add(normalize_url(full_url))
    return links


def extract_content(html: str, url: str, title: str) -> dict:
    """Extract readable text content from a page's HTML."""
    soup = BeautifulSoup(html, BS_PARSER)
    for tag in soup(["script", "style", "noscript", "nav", "footer", "head"]):
        tag.decompose()
    headings = [re.sub(r"\s+", " ", h.get_text(strip=True)) for h in soup.find_all(["h1", "h2", "h3"])]
    body = soup.find("main") or soup.find("article") or soup.find("body")
    raw_text = body.get_text(separator=" ", strip=True) if body else ""
    text = re.sub(r"\s+", " ", raw_text).strip()
    return {
        "url": url,
        "title": title,
        "headings": headings,
        "content": text,
        "crawled_at": datetime.now().isoformat(),
    }

### BFS Queue Crawler

In [ ]:
def content_fingerprint(content: str) -> str:
    """MD5 of first 500 chars — used to detect soft 404s."""
    return hashlib.md5(content[:500].encode()).hexdigest()


def load_robots(start_url: str) -> urllib.robotparser.RobotFileParser | None:
    """Fetch and parse robots.txt for the given site. Returns None on failure."""
    parsed = urlparse(start_url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        return rp
    except Exception:
        return None  # can't read robots.txt — proceed without blocking


async def fetch_with_retry(context, url: str) -> dict:
    """Fetch a page with exponential backoff retry on failure."""
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            return await fetch_page(context, url)
        except Exception as e:
            last_exc = e
            if attempt < MAX_RETRIES - 1:
                await asyncio.sleep(RETRY_BACKOFF ** attempt)
    raise last_exc


async def crawl_site(
    start_url: str,
    max_pages: int = 50,
    concurrency: int = CONCURRENCY_LIMIT,
) -> dict:
    """
    Concurrent BFS crawler using N worker coroutines pulling from a shared asyncio.Queue.
    Respects robots.txt, detects soft 404s, skips binary assets, retries on failure.
    """
    start_url = normalize_url(start_url)

    # --- robots.txt ---
    robots = load_robots(start_url)
    def is_allowed(url: str) -> bool:
        return robots is None or robots.can_fetch(USER_AGENT, url)

    visited: set[str] = set()
    queued_urls: set[str] = {start_url}
    queue: asyncio.Queue[str] = asyncio.Queue()
    pages: list[dict] = []
    invalid_urls: list[dict] = []
    valid_urls: list[dict] = []
    skipped_urls: list[str] = []
    soft_404s: list[str] = []
    lock = asyncio.Lock()

    # Soft 404 fingerprint — set after the start page is successfully fetched
    home_fingerprint: list[str] = []  # single-element list so closure can mutate it

    await queue.put(start_url)
    crawl_start = datetime.now()

    async def worker(context):
        while True:
            url = await queue.get()
            try:
                async with lock:
                    if url in visited:
                        continue
                    # Budget based on successfully extracted pages, not all visited URLs
                    if len(pages) >= max_pages:
                        continue
                    visited.add(url)

                print(f"[{len(pages):>3}/{max_pages}] Crawling: {url}")

                try:
                    result = await fetch_with_retry(context, url)
                    html, title, status = result["html"], result["title"], result["status"]

                    if result.get("skipped"):
                        async with lock:
                            skipped_urls.append(url)
                        continue

                    if status and status >= 400:
                        async with lock:
                            invalid_urls.append({"url": url, "status": status})
                        continue

                    # CPU-bound parsing — outside lock
                    content = extract_content(html, url, title)
                    new_links = extract_links(html, url, start_url)
                    fp = content_fingerprint(content["content"])

                    # Soft 404 detection: pages that silently serve the home page
                    if url == start_url:
                        home_fingerprint.append(fp)
                    elif home_fingerprint and fp == home_fingerprint[0]:
                        async with lock:
                            soft_404s.append(url)
                        print(f"        SOFT 404 (matches home): {url}")
                        continue

                    async with lock:
                        valid_urls.append({"url": url, "status": status})
                        pages.append(content)
                        if len(pages) < max_pages:
                            # Filter robots.txt before queuing
                            to_enqueue = {
                                link for link in new_links - visited - queued_urls
                                if is_allowed(link)
                            }
                            queued_urls.update(to_enqueue)
                        else:
                            to_enqueue = set()

                    for link in to_enqueue:
                        await queue.put(link)

                    elapsed = (datetime.now() - crawl_start).seconds
                    print(f"        title={title!r}  +{len(to_enqueue)} links  queue={queue.qsize()}  {elapsed}s")

                except Exception as e:
                    print(f"        ERROR {url}: {e}")
                    async with lock:
                        invalid_urls.append({"url": url, "error": str(e)})

                finally:
                    await asyncio.sleep(POLITENESS_DELAY)

            finally:
                queue.task_done()

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"],
        )
        context = await browser.new_context(user_agent=USER_AGENT)

        workers = [asyncio.create_task(worker(context)) for _ in range(concurrency)]

        await queue.join()

        for w in workers:
            w.cancel()
        await asyncio.gather(*workers, return_exceptions=True)
        await browser.close()

    total_time = (datetime.now() - crawl_start).total_seconds()
    summary = {
        "crawled_pages": len(pages),
        "valid_urls": len(valid_urls),
        "invalid_urls": len(invalid_urls),
        "soft_404s": len(soft_404s),
        "skipped_assets": len(skipped_urls),
        "total_time_s": round(total_time, 1),
        "pages_per_second": round(len(pages) / total_time, 2) if total_time > 0 else 0,
    }
    print(f"\nDone. {summary}")
    return {
        "data": {
            "pages": pages,
            "valid_urls": valid_urls,
            "invalid_urls": invalid_urls,
            "soft_404s": soft_404s,
            "skipped_urls": skipped_urls,
        },
        "summary": summary,
    }

### Run Crawler

In [ ]:
# nest_asyncio lets asyncio.run() work inside Jupyter's event loop
MAX_PAGES = 500

loop = asyncio.get_event_loop()
crawl_results = loop.run_until_complete(crawl_site(SITE, max_pages=MAX_PAGES))

crawl_results.get("summary")


### Try Out Crawler Module

In [ ]:
from modules.crawler import Crawler

# SET Website to Crawl
# SITE = "https://www.topfaith.edu.ng"
SITE = "https://www.veroliq.com"

# Initialize Crawler
crawl = Crawler()

# Run Crawler
crawl_results = await crawl.run(url=SITE)


summary = crawl_results.get("summary")

summary

In [ ]:
from modules.crawler import Crawler

SITE = "https://www.collectwire.com"
# SITE = "https://www.veroliq.com"
MAX_PAGES = 100

# Initialize Crawler
crawl = Crawler()

print("Running Crawler")

page_count = 1

# Run Crawler
async for page in crawl.run_generator(url=SITE):
    print(f"[{page_count}/{MAX_PAGES}] Pages crawled")
    page_count += 1
    print(page)




In [ ]:
from urllib.parse import urlparse
SITE = "https://www.topfaith.edu.ng"
domain =  (urlparse(SITE).netloc).replace("www.","")

domain

In [ ]:
invalid_urls = crawl_results.get("data").get("invalid_urls")
valid_urls = crawl_results.get("data").get("valid_urls")

print(f"{len(invalid_urls)} Invalid URLs:")
invalid_urls

print(f"{len(valid_urls)} Valid URLs:")
valid_urls



### Inspect Results

In [ ]:
import json

pages = crawl_results.get("data").get("pages")
# Summary table
print(f"{'#':<4} {'URL':<60} {'Title'}")
print("-" * 100)
for i, r in enumerate(pages, 1):
    if "error" in r:
        print(f"{i:<4} {r['url']:<60} ERROR: {r['error'][:40]}")
    else:
        print(f"{i:<4} {r['url']:<60} {r.get('title', '')[:40]}")

In [ ]:
# Inspect a single page's content
page = pages[20]
print(f"URL:      {page['url']}")
print(f"Title:    {page['title']}")
print(f"Headings: {page['headings']}")
print(f"\n--- Content (first 1000 chars) ---\n{page['content'][:1000]}")


### Save Pages to Markdown

In [ ]:
from urllib.parse import urlparse
from modules.crawler import Crawler, save_pages

SITE="https://www.topfaith.edu.ng"

domain = urlparse(SITE).netloc.replace("www.", "")
output_dir = f"data/{domain}"

crawler = Crawler()

results = await crawler.run(url=SITE)

# pages = results["data"]["pages"]



[  1/100] 'Home'  +11 links  q=11  8s
[  6/100] SOFT 404  https://www.topfaith.edu.ng/pg/home
[  2/100] 'Application Form'  +0 links  q=6  11s
[  3/100] 'Careers - Topfaith University'  +0 links  q=4  13s
[  4/100] 'Conferences - Topfaith University'  +0 links  q=4  13s
[  5/100] 'About Us - Topfaith University'  +0 links  q=4  13s
[  6/100] 'Academics'  +5 links  q=8  13s
[  7/100] 'Campus Tour - Topfaith University'  +0 links  q=5  14s
[  8/100] 'Admission - Topfaith University'  +0 links  q=4  15s
[  9/100] 'Fees - Topfaith University'  +0 links  q=4  15s
[ 10/100] 'Topfaith University'  +0 links  q=2  17s
[ 11/100] 'Engineering - Topfaith University'  +8 links  q=9  19s
[ 12/100] 'Management And Social Sciences - Topfaith University'  +5 links  q=14  19s
[ 13/100] 'Law - Topfaith University'  +1 links  q=15  19s
[ 14/100] 'Medical Sciences - Topfaith University'  +4 links  q=19  19s
[ 15/100] 'Engineering - Topfaith University'  +0 links  q=15  21s
[ 16/100] 'Computing And Applied 

NameError: name 'crawl_results' is not defined

In [5]:

pages = results["data"]["pages"]
saved = save_pages(pages, output_dir)
print(f"Saved {saved} pages to {output_dir}/")

Saved 42 pages to data/topfaith.edu.ng/
